# Campaign-Level Performance With Small-Data Uncertainty

## Completed-cycle objective

This notebook evaluates the **12 completed-cycle campaigns** before looking at
individual adsets, audiences, ads, or creatives. It combines:

1. Meta delivery and media-cost totals.
2. Observed WhatsApp orders and revenue.
3. Campaign-type objectives and peer benchmarks.
4. Confidence intervals that show how stable the observed KPIs are.
5. Evidence-aware recommendations for the next cycle.

The notebook does not use an LLM and does not allow uncertainty calculations to
hide the Meta-to-WhatsApp reconciliation limitation. Its purpose is to make every
calculation inspectable by data scientists and business stakeholders.

## 1. Analysis environment

This cell imports the existing `src_2` normalization and deterministic scorecard
services. NumPy is used for reproducible bootstrap resampling, pandas for tables,
and Altair for interval charts. No new statistical dependency is required.

In [1]:
from pathlib import Path
import math

import altair as alt
import numpy as np
import pandas as pd
from IPython.display import Markdown, display

from src_2.analytics import build_assessment_bundle, build_scorecards
from src_2.domain.models import CampaignType, EvidenceStatus
from src_2.infrastructure.configuration import load_campaign_type_registry
from src_2.ingestion import build_data_quality_report, load_sample2, normalize_cycle

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_colwidth", 140)
alt.data_transformers.disable_max_rows()

CONFIDENCE_LEVEL = 0.95
Z_95 = 1.959963984540054
N_BOOTSTRAP = 5_000
RANDOM_SEED = 42

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "src_2").exists():
    raise RuntimeError("Run this notebook from the Team2-MarketingExpert repository root.")

INPUT_DIRECTORY = REPO_ROOT / "src_2" / "data" / "input" / "sampe_2"
print(f"Repository: {REPO_ROOT}")
print(f"Input data: {INPUT_DIRECTORY}")
print(f"Bootstrap repetitions: {N_BOOTSTRAP:,}")

Repository: /Users/abdelmoo/Desktop/CAPI Analysis/Team2-MarketingExpert
Input data: /Users/abdelmoo/Desktop/CAPI Analysis/Team2-MarketingExpert/src_2/data/input/sampe_2
Bootstrap repetitions: 5,000


## 2. Normalize the completed cycle

Normalization creates separate tables at their natural grain: campaign, adset,
ad, creative, daily Meta insight, conversation, order line, and product. Raw
messages and direct personal details are deliberately excluded from this analytic
layer. Keeping the facts separate prevents spend or revenue from being duplicated.

In [2]:
raw_payload = load_sample2(INPUT_DIRECTORY)
canonical = normalize_cycle(raw_payload)

inventory = pd.DataFrame(
    {
        "Canonical table": [
            "campaigns", "adsets", "ads", "creatives",
            "media_daily", "conversations", "order_lines", "products",
        ],
        "Rows": [
            len(canonical.campaigns), len(canonical.adsets), len(canonical.ads),
            len(canonical.creatives), len(canonical.media_daily),
            len(canonical.conversations), len(canonical.order_lines),
            len(canonical.products),
        ],
        "Natural grain": [
            "campaign", "adset", "ad", "creative", "ad-day",
            "conversation", "order-product line", "product",
        ],
    }
)
display(inventory)

,Canonical table,Rows,Natural grain
0,campaigns,12,campaign
1,adsets,26,adset
2,ads,40,ad
3,creatives,30,creative
4,media_daily,1243,ad-day
5,conversations,788,conversation
6,order_lines,1081,order-product line
7,products,105,product


## 3. Establish the data-quality boundary

Meta conversation starts and supplied WhatsApp records are counted separately.
Their ratio is a reconciliation diagnostic, not automatically literal coverage.
Confidence intervals can quantify random variation inside the supplied WhatsApp
sample; they cannot correct missing, selectively exported, or differently defined
conversations.

In [3]:
quality = build_data_quality_report(canonical)

quality_table = pd.DataFrame(
    {
        "Check": [
            "Evidence status",
            "Meta-attributed conversation starts",
            "Observed Meta-sourced WhatsApp conversations",
            "Reconciliation ratio",
            "Event definitions reconciled",
            "Unmatched campaign references",
            "Unmatched adset references",
            "Unmatched ad references",
        ],
        "Value": [
            quality.status.value,
            f"{quality.meta_conversation_starts:,}",
            f"{quality.observed_meta_whatsapp_conversations:,}",
            f"{quality.reconciliation_ratio:.2%}" if quality.reconciliation_ratio is not None else "Unavailable",
            quality.event_definitions_reconciled,
            quality.unmatched_campaigns,
            quality.unmatched_adsets,
            quality.unmatched_ads,
        ],
    }
)
display(quality_table)
for warning in quality.warnings:
    print(f"WARNING: {warning}")

,Check,Value
0,Evidence status,limited_evidence
1,Meta-attributed conversation starts,"116,098"
2,Observed Meta-sourced WhatsApp conversations,617
3,Reconciliation ratio,0.53%
4,Event definitions reconciled,False
5,Unmatched campaign references,0
6,Unmatched adset references,0
7,Unmatched ad references,0


## 4. Build the campaign scorecard

Media, conversations, products, and setup counts are aggregated independently and
joined only after aggregation. KPIs are calculated from total numerators and
denominators rather than averaging daily ratios. The same service also builds
adset, ad, creative, and audience scorecards, but this notebook intentionally uses
only the campaign table.

In [4]:
raw_scorecards = build_scorecards(canonical)
registry = load_campaign_type_registry()
cycle_id = f"cycle_{canonical.cycle_start.date()}_{canonical.cycle_end.date()}"
assessment_bundle = build_assessment_bundle(
    cycle_id, raw_scorecards, registry, quality
)
campaigns = assessment_bundle.scorecards.campaign.copy()

point_columns = [
    "campaign_name", "campaign_type", "active_days", "spend",
    "impressions", "link_ctr_pct", "meta_conversation_starts",
    "observed_conversations", "orders_created", "delivered_orders",
    "net_revenue", "net_roas", "aov", "delivered_rate",
    "negative_outcome_rate",
]
display(campaigns[point_columns].sort_values(["campaign_type", "campaign_name"]))

,campaign_name,campaign_type,active_days,spend,impressions,link_ctr_pct,meta_conversation_starts,observed_conversations,orders_created,delivered_orders,net_revenue,net_roas,aov,delivered_rate,negative_outcome_rate
0,Always-On Premium Acquisition,always_on,180,140783.91,26091014,0.754283,26855.0,111,70,52,65824.0,0.467553,1221.038462,0.468468,0.468468
1,Awareness Boost January,awareness,31,5966.66,1089878,1.195822,1621.0,4,2,1,1096.0,0.183687,1096.000000,0.250000,0.750000
3,January Trial Bundle Test,experimental,27,2817.50,532636,1.275543,852.0,15,8,5,3004.0,1.066193,541.600000,0.333333,0.666667
7,Post-Eid Lookalike Test,experimental,24,7762.12,1863544,0.879883,1645.0,63,42,27,42145.0,5.429573,1534.555556,0.428571,0.396825
8,Summer Premium Launch,launch,60,68595.84,14187459,1.143468,21236.0,97,77,64,70521.0,1.028065,1086.843750,0.659794,0.268041
10,Mid-Year Sale,promotional,22,17552.49,3378126,1.458501,7238.0,34,30,20,23431.0,1.334910,817.900000,0.588235,0.411765
2,Pre-Ramadan Bundle Promo,promotional,27,13958.72,2872008,1.340491,5426.0,46,32,21,28905.0,2.070749,1225.238095,0.456522,0.478261
11,Summer Retention Push,retention,23,9006.65,1806267,1.229497,3856.0,16,10,7,4722.0,0.524279,674.571429,0.437500,0.500000
9,Lookalike Scale Cycle 3,scale,48,31465.87,6206354,1.130922,9473.0,36,27,23,37936.0,1.205624,1635.304348,0.638889,0.305556
6,Eid Gifting Premium,seasonal,15,14976.24,2967717,1.460517,6255.0,47,41,28,64990.0,4.339540,2234.857143,0.595745,0.382979


## 5. Understand the uncertainty methods

The point estimates above describe exactly what appears in the supplied cycle.
Intervals answer how stable those patterns are under explicit assumptions:

- **Wilson 95% interval:** binary conversation outcomes such as delivered/not
  delivered. It behaves better than the simple normal interval for small samples.
- **Percentile bootstrap:** repeatedly resamples observed conversations and
  recalculates monetary KPIs. It measures sensitivity to the observed customer mix.
- **No interval for exact counts or Meta reach:** totals such as spend and observed
  conversation count are reported exactly. A valid future-count interval requires a
  time-series/count model, while period reach requires a deduplicated Meta estimate.

These are uncertainty summaries, not proof of causality or a correction for sample
selection bias.

In [5]:
def wilson_interval(successes, trials, z=Z_95):
    # Return a Wilson score interval for a binomial proportion.
    if trials is None or trials <= 0:
        return np.nan, np.nan
    successes = float(successes)
    trials = float(trials)
    proportion = successes / trials
    denominator = 1 + z**2 / trials
    center = (proportion + z**2 / (2 * trials)) / denominator
    half_width = (
        z
        * math.sqrt(
            proportion * (1 - proportion) / trials
            + z**2 / (4 * trials**2)
        )
        / denominator
    )
    return max(0.0, center - half_width), min(1.0, center + half_width)


def percentile_interval(values, alpha=0.05):
    values = np.asarray(values, dtype=float)
    values = values[np.isfinite(values)]
    if values.size == 0:
        return np.nan, np.nan
    return tuple(np.quantile(values, [alpha / 2, 1 - alpha / 2]))


def format_interval(lower, upper, *, percent=False, money=False, ratio=False):
    if pd.isna(lower) or pd.isna(upper):
        return "Not estimated"
    if percent:
        return f"{lower:.1%} to {upper:.1%}"
    if money:
        return f"EGP {lower:,.0f} to EGP {upper:,.0f}"
    if ratio:
        return f"{lower:.2f} to {upper:.2f}"
    return f"{lower:,.2f} to {upper:,.2f}"


# Sanity checks that make the small-sample behavior visible.
for successes, trials in [(52, 111), (2, 3), (0, 3)]:
    lower, upper = wilson_interval(successes, trials)
    print(
        f"{successes}/{trials} = {successes/trials:.1%}; "
        f"95% Wilson interval {lower:.1%} to {upper:.1%}"
    )

52/111 = 46.8%; 95% Wilson interval 37.8% to 56.1%
2/3 = 66.7%; 95% Wilson interval 20.8% to 93.9%
0/3 = 0.0%; 95% Wilson interval 0.0% to 56.1%


## 6. Calculate confidence intervals for conversation rates

Each conversation is treated as one binary trial for the selected outcome. The
denominator is always shown because an impressive percentage supported by three
conversations should not be interpreted like the same percentage supported by one
hundred conversations.

In [6]:
rate_specs = {
    "order_creation_rate": ("orders_created", "observed_conversations"),
    "delivered_rate": ("delivered_orders", "observed_conversations"),
    "negative_outcome_rate": ("negative_outcomes", "observed_conversations"),
    "repeat_conversation_rate": ("repeat_conversations", "observed_conversations"),
    "repeat_order_rate": ("repeat_delivered_orders", "delivered_orders"),
    "refund_rate": (
        "refunded_orders",
        "delivered_plus_refunded",
    ),
}

rate_rows = []
for _, campaign in campaigns.iterrows():
    row = {"campaign_id": campaign["campaign_id"]}
    for metric, (numerator_col, denominator_col) in rate_specs.items():
        numerator = int(campaign[numerator_col])
        denominator = (
            int(campaign["delivered_orders"] + campaign["refunded_orders"])
            if denominator_col == "delivered_plus_refunded"
            else int(campaign[denominator_col])
        )
        lower, upper = wilson_interval(numerator, denominator)
        row[f"{metric}_numerator"] = numerator
        row[f"{metric}_denominator"] = denominator
        row[f"{metric}_lower"] = lower
        row[f"{metric}_upper"] = upper
    rate_rows.append(row)

rate_uncertainty = pd.DataFrame(rate_rows)
campaign_uncertainty = campaigns.merge(rate_uncertainty, on="campaign_id", how="left")

rate_view = campaign_uncertainty[
    [
        "campaign_name", "campaign_type", "observed_conversations",
        "delivered_rate", "delivered_rate_lower", "delivered_rate_upper",
        "negative_outcome_rate", "negative_outcome_rate_lower",
        "negative_outcome_rate_upper",
    ]
].copy()
display(rate_view.sort_values("delivered_rate", ascending=False))

,campaign_name,campaign_type,observed_conversations,delivered_rate,delivered_rate_lower,delivered_rate_upper,negative_outcome_rate,negative_outcome_rate_lower,negative_outcome_rate_upper
8,Summer Premium Launch,launch,97,0.659794,0.561036,0.746377,0.268041,0.189976,0.363779
4,Ramadan Suhoor Specials,seasonal,64,0.640625,0.518209,0.747116,0.328125,0.225706,0.450009
9,Lookalike Scale Cycle 3,scale,36,0.638889,0.475751,0.775244,0.305556,0.180045,0.468563
6,Eid Gifting Premium,seasonal,47,0.595745,0.453421,0.723600,0.382979,0.257907,0.525734
10,Mid-Year Sale,promotional,34,0.588235,0.422216,0.736340,0.411765,0.263660,0.577784
5,Ramadan Iftar Premium Bundles,seasonal,84,0.500000,0.395439,0.604561,0.404762,0.306196,0.511658
0,Always-On Premium Acquisition,always_on,111,0.468468,0.378252,0.560794,0.468468,0.378252,0.560794
2,Pre-Ramadan Bundle Promo,promotional,46,0.456522,0.321547,0.598198,0.478261,0.341247,0.618626
11,Summer Retention Push,retention,16,0.437500,0.230987,0.668214,0.500000,0.279996,0.720004
7,Post-Eid Lookalike Test,experimental,63,0.428571,0.313969,0.551384,0.396825,0.285319,0.520191


## 7. Bootstrap monetary KPI stability

For each campaign, the observed conversations are sampled with replacement 5,000
times. Every resample recalculates net revenue, net ROAS, delivered-order cost,
AOV, and net revenue per active day. Spend and the number of observed conversations
are held fixed, so these intervals describe sensitivity to **customer outcome mix**;
they do not include uncertainty about future media cost or future conversation
volume.

In [7]:
def bootstrap_campaign(conversations, spend, period_days, *, seed):
    if conversations.empty:
        return {}

    rng = np.random.default_rng(seed)
    n = len(conversations)
    indices = rng.integers(0, n, size=(N_BOOTSTRAP, n))

    net_revenue = conversations["net_revenue"].to_numpy(dtype=float)[indices].sum(axis=1)
    delivered_revenue = conversations["delivered_revenue"].to_numpy(dtype=float)[indices].sum(axis=1)
    delivered_orders = conversations["is_delivered"].to_numpy(dtype=float)[indices].sum(axis=1)

    with np.errstate(divide="ignore", invalid="ignore"):
        net_roas = net_revenue / spend if spend > 0 else np.full(N_BOOTSTRAP, np.nan)
        delivered_roas = delivered_revenue / spend if spend > 0 else np.full(N_BOOTSTRAP, np.nan)
        cost_per_delivered_order = np.where(delivered_orders > 0, spend / delivered_orders, np.nan)
        aov = np.where(delivered_orders > 0, delivered_revenue / delivered_orders, np.nan)
        net_revenue_per_day = net_revenue / max(float(period_days), 1.0)

    metrics = {
        "net_revenue": net_revenue,
        "net_roas": net_roas,
        "delivered_roas": delivered_roas,
        "cost_per_delivered_order": cost_per_delivered_order,
        "aov": aov,
        "net_revenue_per_day": net_revenue_per_day,
    }
    output = {}
    for metric, values in metrics.items():
        lower, upper = percentile_interval(values)
        output[f"{metric}_bootstrap_lower"] = lower
        output[f"{metric}_bootstrap_upper"] = upper
    return output


bootstrap_rows = []
for position, campaign in campaigns.reset_index(drop=True).iterrows():
    campaign_conversations = canonical.conversations[
        canonical.conversations["campaign_id"].eq(campaign["campaign_id"])
    ]
    result = bootstrap_campaign(
        campaign_conversations,
        float(campaign["spend"]),
        float(campaign["period_days"]),
        seed=RANDOM_SEED + position,
    )
    bootstrap_rows.append({"campaign_id": campaign["campaign_id"], **result})

bootstrap_uncertainty = pd.DataFrame(bootstrap_rows)
campaign_uncertainty = campaign_uncertainty.merge(
    bootstrap_uncertainty, on="campaign_id", how="left"
)

monetary_view = campaign_uncertainty[
    [
        "campaign_name", "campaign_type", "observed_conversations",
        "net_roas", "net_roas_bootstrap_lower", "net_roas_bootstrap_upper",
        "aov", "aov_bootstrap_lower", "aov_bootstrap_upper",
    ]
].copy()
display(monetary_view.sort_values("net_roas", ascending=False))

,campaign_name,campaign_type,observed_conversations,net_roas,net_roas_bootstrap_lower,net_roas_bootstrap_upper,aov,aov_bootstrap_lower,aov_bootstrap_upper
7,Post-Eid Lookalike Test,experimental,63,5.429573,3.627340,7.354067,1534.555556,1244.926724,1842.732500
6,Eid Gifting Premium,seasonal,47,4.339540,3.015034,5.798316,2234.857143,1700.386329,2829.787500
2,Pre-Ramadan Bundle Promo,promotional,46,2.070749,1.351977,2.887827,1225.238095,961.580952,1533.645455
5,Ramadan Iftar Premium Bundles,seasonal,84,1.348588,1.017734,1.701751,1770.404762,1520.710432,2047.797983
10,Mid-Year Sale,promotional,34,1.334910,0.934048,1.783519,817.900000,613.187500,1050.944118
9,Lookalike Scale Cycle 3,scale,36,1.205624,0.668802,1.900678,1635.304348,984.985000,2489.859278
4,Ramadan Suhoor Specials,seasonal,64,1.176774,0.893367,1.507864,910.073171,752.951220,1109.866647
3,January Trial Bundle Test,experimental,15,1.066193,0.430878,1.771074,541.600000,480.000000,649.333333
8,Summer Premium Launch,launch,97,1.028065,0.801129,1.282368,1086.843750,889.427857,1319.109242
11,Summer Retention Push,retention,16,0.524279,0.227543,0.864078,674.571429,484.264286,888.508929


## 8. Visualize delivered-rate uncertainty

The point is the observed delivered rate. The horizontal line is its Wilson 95%
interval. Long lines indicate that the available conversations do not locate the
underlying rate precisely. Campaign color represents campaign type; color does not
change the statistical calculation.

In [8]:
delivered_chart_data = campaign_uncertainty[
    [
        "campaign_name", "campaign_type", "observed_conversations",
        "delivered_rate", "delivered_rate_lower", "delivered_rate_upper",
    ]
].copy()

delivered_order = delivered_chart_data.sort_values("delivered_rate")["campaign_name"].tolist()
delivered_base = alt.Chart(delivered_chart_data).encode(
    y=alt.Y("campaign_name:N", sort=delivered_order, title=None),
    color=alt.Color("campaign_type:N", title="Campaign type"),
    tooltip=[
        alt.Tooltip("campaign_name:N", title="Campaign"),
        alt.Tooltip("campaign_type:N", title="Type"),
        alt.Tooltip("observed_conversations:Q", title="Observed conversations", format=",.0f"),
        alt.Tooltip("delivered_rate:Q", title="Delivered rate", format=".1%"),
        alt.Tooltip("delivered_rate_lower:Q", title="95% lower", format=".1%"),
        alt.Tooltip("delivered_rate_upper:Q", title="95% upper", format=".1%"),
    ],
)

delivered_intervals = delivered_base.mark_rule(strokeWidth=3).encode(
    x=alt.X("delivered_rate_lower:Q", title="Delivered rate with Wilson 95% interval", axis=alt.Axis(format="%")),
    x2="delivered_rate_upper:Q",
)
delivered_points = delivered_base.mark_point(filled=True, size=95, stroke="white", strokeWidth=1).encode(
    x=alt.X("delivered_rate:Q", axis=alt.Axis(format="%"))
)

display((delivered_intervals + delivered_points).properties(width=760, height=360))

alt.LayerChart(...)

## 9. Visualize observed net ROAS stability

This chart uses the conversation bootstrap interval. A vertical line at ROAS 1.0
is a revenue-versus-media-spend reference, not a profit break-even line: product,
delivery, tax, staffing, and overhead costs are unavailable. The result also covers
only the supplied WhatsApp outcomes.

In [9]:
roas_chart_data = campaign_uncertainty.dropna(
    subset=["net_roas_bootstrap_lower", "net_roas_bootstrap_upper"]
)[
    [
        "campaign_name", "campaign_type", "observed_conversations",
        "net_roas", "net_roas_bootstrap_lower", "net_roas_bootstrap_upper",
    ]
].copy()
roas_order = roas_chart_data.sort_values("net_roas")["campaign_name"].tolist()
roas_base = alt.Chart(roas_chart_data).encode(
    y=alt.Y("campaign_name:N", sort=roas_order, title=None),
    color=alt.Color("campaign_type:N", title="Campaign type"),
    tooltip=[
        alt.Tooltip("campaign_name:N", title="Campaign"),
        alt.Tooltip("campaign_type:N", title="Type"),
        alt.Tooltip("observed_conversations:Q", title="Observed conversations", format=",.0f"),
        alt.Tooltip("net_roas:Q", title="Observed net ROAS", format=".2f"),
        alt.Tooltip("net_roas_bootstrap_lower:Q", title="95% lower", format=".2f"),
        alt.Tooltip("net_roas_bootstrap_upper:Q", title="95% upper", format=".2f"),
    ],
)
roas_intervals = roas_base.mark_rule(strokeWidth=3).encode(
    x=alt.X("net_roas_bootstrap_lower:Q", title="Observed net ROAS with bootstrap 95% interval"),
    x2="net_roas_bootstrap_upper:Q",
)
roas_points = roas_base.mark_point(filled=True, size=95, stroke="white", strokeWidth=1).encode(
    x="net_roas:Q"
)
reference = alt.Chart(pd.DataFrame({"x": [1.0]})).mark_rule(
    color="#555555", strokeDash=[5, 4]
).encode(x="x:Q")

display((roas_intervals + roas_points + reference).properties(width=760, height=360))

alt.LayerChart(...)

## 10. Connect campaign type, primary KPI, and benchmark

Each campaign is evaluated against the objective of its campaign type. The current
POC benchmark excludes the campaign itself, then uses the current-cycle same-type
median when another campaign of that type exists and the portfolio median otherwise.
These are comparison references, not approved business targets.

When a compatible interval exists, the notebook asks whether the whole interval is
above or below the benchmark. When an interval cannot be justified from the supplied
data, the result is explicitly marked `point estimate only`.

In [10]:
assessments_by_id = {
    item.campaign_id: item for item in assessment_bundle.assessments
}
packs_by_id = {
    item.campaign_id: item for item in assessment_bundle.evidence_packs
}

primary_interval_columns = {
    "net_roas": ("net_roas_bootstrap_lower", "net_roas_bootstrap_upper"),
    "net_revenue": ("net_revenue_bootstrap_lower", "net_revenue_bootstrap_upper"),
    "net_revenue_per_day": (
        "net_revenue_per_day_bootstrap_lower",
        "net_revenue_per_day_bootstrap_upper",
    ),
}


def statistical_conclusion(actual, lower, upper, benchmark, direction):
    if benchmark is None or pd.isna(benchmark):
        return "No compatible peer benchmark"
    if pd.isna(lower) or pd.isna(upper):
        return "Point estimate only; uncertainty interval not estimated"
    if direction == "higher":
        if lower > benchmark:
            return "Credible observed outperformer"
        if upper < benchmark:
            return "Credible observed underperformer"
    else:
        if upper < benchmark:
            return "Credible observed outperformer"
        if lower > benchmark:
            return "Credible observed underperformer"
    return "Inconclusive relative to the POC benchmark"


def uncertainty_recommendation(conclusion, evidence_status, current_action):
    if evidence_status == EvidenceStatus.DATA_NOT_READY.value:
        return "Data not ready"
    if evidence_status == EvidenceStatus.INSUFFICIENT.value:
        return "Insufficient evidence; collect more observations"
    if conclusion == "Credible observed outperformer":
        if evidence_status == EvidenceStatus.READY.value:
            return "Scale candidate, subject to business approval"
        return "Keep as test; scale candidate after data reconciliation"
    if conclusion == "Credible observed underperformer":
        return "Do not increase funding unchanged; diagnose and retest"
    if conclusion.startswith("Inconclusive"):
        return "Keep as controlled test; evidence does not separate performance"
    return f"Manual review; current deterministic action is {current_action}"


decision_rows = []
for _, campaign in campaign_uncertainty.iterrows():
    campaign_id = str(campaign["campaign_id"])
    pack = packs_by_id[campaign_id]
    assessment = assessments_by_id[campaign_id]
    primary = pack.primary_kpis[0]
    interval_columns = primary_interval_columns.get(primary.metric)
    lower = campaign[interval_columns[0]] if interval_columns else np.nan
    upper = campaign[interval_columns[1]] if interval_columns else np.nan
    conclusion = statistical_conclusion(
        primary.actual, lower, upper, primary.benchmark, primary.direction
    )
    recommendation = uncertainty_recommendation(
        conclusion,
        assessment.evidence_status.value,
        assessment.next_cycle_action.value,
    )
    config = registry.campaign_types[CampaignType(campaign["campaign_type"])]

    decision_rows.append(
        {
            "campaign_id": campaign_id,
            "Campaign": campaign["campaign_name"],
            "Type": campaign["campaign_type"],
            "Business objective": config.business_job,
            "Primary KPI": primary.label,
            "Primary value": primary.actual,
            "Primary CI lower": lower,
            "Primary CI upper": upper,
            "POC benchmark": primary.benchmark,
            "Benchmark source": primary.benchmark_source,
            "Statistical conclusion": conclusion,
            "Observed conversations": int(campaign["observed_conversations"]),
            "Delivered orders": int(campaign["delivered_orders"]),
            "Spend": float(campaign["spend"]),
            "Observed net revenue": float(campaign["net_revenue"]),
            "Observed net ROAS": float(campaign["net_roas"]) if pd.notna(campaign["net_roas"]) else np.nan,
            "Delivered rate": float(campaign["delivered_rate"]) if pd.notna(campaign["delivered_rate"]) else np.nan,
            "Delivered CI lower": campaign["delivered_rate_lower"],
            "Delivered CI upper": campaign["delivered_rate_upper"],
            "Rule target result": assessment.target_status.value,
            "Rule funding action": assessment.next_cycle_action.value,
            "Uncertainty-aware recommendation": recommendation,
            "Evidence status": assessment.evidence_status.value,
        }
    )

campaign_decisions = pd.DataFrame(decision_rows)
display(
    campaign_decisions[
        [
            "Campaign", "Type", "Primary KPI", "Primary value",
            "Primary CI lower", "Primary CI upper", "POC benchmark",
            "Statistical conclusion", "Rule funding action",
            "Uncertainty-aware recommendation", "Evidence status",
        ]
    ].sort_values(["Type", "Campaign"])
)

,Campaign,Type,Primary KPI,Primary value,Primary CI lower,Primary CI upper,POC benchmark,Statistical conclusion,Rule funding action,Uncertainty-aware recommendation,Evidence status
0,Always-On Premium Acquisition,always_on,Net return on ad spend,0.467553,0.330656,0.628004,1.205624e+00,Credible observed underperformer,do_not_fund,Do not increase funding unchanged; diagnose and retest,limited_evidence
1,Awareness Boost January,awareness,Reach,636884.000000,NaN,NaN,2.598221e+06,Point estimate only; uncertainty interval not estimated,keep_as_test,Manual review; current deterministic action is keep_as_test,limited_evidence
3,January Trial Bundle Test,experimental,Observed WhatsApp conversations,15.000000,NaN,NaN,6.300000e+01,Point estimate only; uncertainty interval not estimated,keep_as_test,Manual review; current deterministic action is keep_as_test,limited_evidence
7,Post-Eid Lookalike Test,experimental,Observed WhatsApp conversations,63.000000,NaN,NaN,1.500000e+01,Point estimate only; uncertainty interval not estimated,keep_as_test,Manual review; current deterministic action is keep_as_test,limited_evidence
8,Summer Premium Launch,launch,Net revenue,70521.000000,54954.100000,87965.100000,3.793600e+04,Credible observed outperformer,keep_as_test,Keep as test; scale candidate after data reconciliation,limited_evidence
10,Mid-Year Sale,promotional,Net return on ad spend,1.334910,0.934048,1.783519,2.070749e+00,Credible observed underperformer,keep_as_test,Do not increase funding unchanged; diagnose and retest,limited_evidence
2,Pre-Ramadan Bundle Promo,promotional,Net return on ad spend,2.070749,1.351977,2.887827,1.334910e+00,Credible observed outperformer,keep_as_test,Keep as test; scale candidate after data reconciliation,limited_evidence
11,Summer Retention Push,retention,Repeat delivered orders,7.000000,NaN,NaN,2.300000e+01,Point estimate only; uncertainty interval not estimated,do_not_fund,Manual review; current deterministic action is do_not_fund,limited_evidence
9,Lookalike Scale Cycle 3,scale,Observed WhatsApp conversations,36.000000,NaN,NaN,4.700000e+01,Point estimate only; uncertainty interval not estimated,keep_as_test,Manual review; current deterministic action is keep_as_test,limited_evidence
6,Eid Gifting Premium,seasonal,Net revenue per active day,4332.666667,3010.258333,5789.131667,1.916667e+03,Credible observed outperformer,keep_as_test,Keep as test; scale candidate after data reconciliation,limited_evidence


## 11. Whole-cycle campaign decision table

This is the compact management view: one row per campaign. It keeps the primary
objective, point estimate, uncertainty, benchmark, observed business totals, current
deterministic rule, and uncertainty-aware interpretation together. `Not estimated`
is preferable to manufacturing a misleading interval for reach or exact cycle counts.

In [11]:
summary = campaign_decisions.copy()
summary["Primary 95% interval"] = summary.apply(
    lambda row: format_interval(
        row["Primary CI lower"], row["Primary CI upper"], ratio=row["Primary KPI"] == "Net return on ad spend"
    ),
    axis=1,
)
summary["Delivered rate (95% interval)"] = summary.apply(
    lambda row: (
        f"{row['Delivered rate']:.1%} "
        f"[{row['Delivered CI lower']:.1%}, {row['Delivered CI upper']:.1%}]"
    ),
    axis=1,
)
summary["Spend (EGP)"] = summary["Spend"].map(lambda value: f"{value:,.0f}")
summary["Observed net revenue (EGP)"] = summary["Observed net revenue"].map(lambda value: f"{value:,.0f}")
summary["Observed net ROAS"] = summary["Observed net ROAS"].map(
    lambda value: f"{value:.2f}" if pd.notna(value) else "Unavailable"
)

summary_columns = [
    "Campaign", "Type", "Primary KPI", "Primary value",
    "Primary 95% interval", "POC benchmark", "Statistical conclusion",
    "Observed conversations", "Delivered orders",
    "Delivered rate (95% interval)", "Spend (EGP)",
    "Observed net revenue (EGP)", "Observed net ROAS",
    "Rule target result", "Rule funding action",
    "Uncertainty-aware recommendation", "Evidence status",
]
display(summary[summary_columns].sort_values(["Type", "Campaign"]).reset_index(drop=True))

,Campaign,Type,Primary KPI,Primary value,Primary 95% interval,POC benchmark,Statistical conclusion,Observed conversations,Delivered orders,Delivered rate (95% interval),Spend (EGP),Observed net revenue (EGP),Observed net ROAS,Rule target result,Rule funding action,Uncertainty-aware recommendation,Evidence status
0,Always-On Premium Acquisition,always_on,Net return on ad spend,0.467553,0.33 to 0.63,1.205624e+00,Credible observed underperformer,111,52,"46.8% [37.8%, 56.1%]","140,784","65,824",0.47,not_achieved,do_not_fund,Do not increase funding unchanged; diagnose and retest,limited_evidence
1,Awareness Boost January,awareness,Reach,636884.000000,Not estimated,2.598221e+06,Point estimate only; uncertainty interval not estimated,4,1,"25.0% [4.6%, 69.9%]","5,967","1,096",0.18,not_achieved,keep_as_test,Manual review; current deterministic action is keep_as_test,limited_evidence
2,January Trial Bundle Test,experimental,Observed WhatsApp conversations,15.000000,Not estimated,6.300000e+01,Point estimate only; uncertainty interval not estimated,15,5,"33.3% [15.2%, 58.3%]","2,818","3,004",1.07,not_achieved,keep_as_test,Manual review; current deterministic action is keep_as_test,limited_evidence
3,Post-Eid Lookalike Test,experimental,Observed WhatsApp conversations,63.000000,Not estimated,1.500000e+01,Point estimate only; uncertainty interval not estimated,63,27,"42.9% [31.4%, 55.1%]","7,762","42,145",5.43,achieved_with_concerns,keep_as_test,Manual review; current deterministic action is keep_as_test,limited_evidence
4,Summer Premium Launch,launch,Net revenue,70521.000000,"54,954.10 to 87,965.10",3.793600e+04,Credible observed outperformer,97,64,"66.0% [56.1%, 74.6%]","68,596","70,521",1.03,achieved_with_concerns,keep_as_test,Keep as test; scale candidate after data reconciliation,limited_evidence
5,Mid-Year Sale,promotional,Net return on ad spend,1.334910,0.93 to 1.78,2.070749e+00,Credible observed underperformer,34,20,"58.8% [42.2%, 73.6%]","17,552","23,431",1.33,not_achieved,keep_as_test,Do not increase funding unchanged; diagnose and retest,limited_evidence
6,Pre-Ramadan Bundle Promo,promotional,Net return on ad spend,2.070749,1.35 to 2.89,1.334910e+00,Credible observed outperformer,46,21,"45.7% [32.2%, 59.8%]","13,959","28,905",2.07,achieved_with_concerns,keep_as_test,Keep as test; scale candidate after data reconciliation,limited_evidence
7,Summer Retention Push,retention,Repeat delivered orders,7.000000,Not estimated,2.300000e+01,Point estimate only; uncertainty interval not estimated,16,7,"43.8% [23.1%, 66.8%]","9,007","4,722",0.52,not_achieved,do_not_fund,Manual review; current deterministic action is do_not_fund,limited_evidence
8,Lookalike Scale Cycle 3,scale,Observed WhatsApp conversations,36.000000,Not estimated,4.700000e+01,Point estimate only; uncertainty interval not estimated,36,23,"63.9% [47.6%, 77.5%]","31,466","37,936",1.21,not_achieved,keep_as_test,Manual review; current deterministic action is keep_as_test,limited_evidence
9,Eid Gifting Premium,seasonal,Net revenue per active day,4332.666667,"3,010.26 to 5,789.13",1.916667e+03,Credible observed outperformer,47,28,"59.6% [45.3%, 72.4%]","14,976","64,990",4.34,achieved_with_concerns,keep_as_test,Keep as test; scale candidate after data reconciliation,limited_evidence


## 12. Detailed table for every campaign

The final output below prints one business-readable table per campaign. Each table
separates the objective, evidence volume, point estimates, uncertainty, current
rules, and the uncertainty-aware recommendation. This is the intended review format
before drilling into adsets, audiences, ads, and creatives.

In [12]:
uncertainty_by_id = campaign_uncertainty.set_index("campaign_id")

for _, decision in campaign_decisions.sort_values(["Type", "Campaign"]).iterrows():
    campaign = uncertainty_by_id.loc[decision["campaign_id"]]
    primary_interval = format_interval(
        decision["Primary CI lower"],
        decision["Primary CI upper"],
        ratio=decision["Primary KPI"] == "Net return on ad spend",
    )
    delivered_interval = format_interval(
        campaign["delivered_rate_lower"],
        campaign["delivered_rate_upper"],
        percent=True,
    )
    order_interval = format_interval(
        campaign["order_creation_rate_lower"],
        campaign["order_creation_rate_upper"],
        percent=True,
    )
    negative_interval = format_interval(
        campaign["negative_outcome_rate_lower"],
        campaign["negative_outcome_rate_upper"],
        percent=True,
    )
    roas_interval = format_interval(
        campaign["net_roas_bootstrap_lower"],
        campaign["net_roas_bootstrap_upper"],
        ratio=True,
    )
    aov_interval = format_interval(
        campaign["aov_bootstrap_lower"],
        campaign["aov_bootstrap_upper"],
        money=True,
    )

    display(Markdown(f"### {decision['Campaign']}"))
    detail = pd.DataFrame(
        {
            "Field": [
                "Campaign type",
                "Business objective",
                "Primary KPI",
                "Primary KPI value",
                "Primary KPI 95% interval",
                "POC benchmark",
                "Benchmark source",
                "Statistical conclusion",
                "Observed conversations",
                "Order-creation rate",
                "Order-creation 95% interval",
                "Delivered orders",
                "Delivered rate",
                "Delivered-rate 95% interval",
                "Negative-outcome rate",
                "Negative-outcome 95% interval",
                "Spend",
                "Observed net revenue",
                "Observed net ROAS",
                "Observed net ROAS 95% interval",
                "AOV",
                "AOV 95% interval",
                "Current target result",
                "Current funding action",
                "Uncertainty-aware recommendation",
                "Evidence status",
            ],
            "Value": [
                decision["Type"],
                decision["Business objective"],
                decision["Primary KPI"],
                f"{decision['Primary value']:,.3f}" if pd.notna(decision["Primary value"]) else "Unavailable",
                primary_interval,
                f"{decision['POC benchmark']:,.3f}" if pd.notna(decision["POC benchmark"]) else "Unavailable",
                decision["Benchmark source"] or "Unavailable",
                decision["Statistical conclusion"],
                f"{int(decision['Observed conversations']):,}",
                f"{campaign['order_creation_rate']:.1%}",
                order_interval,
                f"{int(decision['Delivered orders']):,}",
                f"{campaign['delivered_rate']:.1%}",
                delivered_interval,
                f"{campaign['negative_outcome_rate']:.1%}",
                negative_interval,
                f"EGP {decision['Spend']:,.0f}",
                f"EGP {decision['Observed net revenue']:,.0f}",
                f"{decision['Observed net ROAS']:.2f}" if pd.notna(decision["Observed net ROAS"]) else "Unavailable",
                roas_interval,
                f"EGP {campaign['aov']:,.0f}" if pd.notna(campaign["aov"]) else "Unavailable",
                aov_interval,
                decision["Rule target result"],
                decision["Rule funding action"],
                decision["Uncertainty-aware recommendation"],
                decision["Evidence status"],
            ],
        }
    )
    display(detail)

### Always-On Premium Acquisition

,Field,Value
0,Campaign type,always_on
1,Business objective,Acquire steady sales with stable economics.
2,Primary KPI,Net return on ad spend
3,Primary KPI value,0.468
4,Primary KPI 95% interval,0.33 to 0.63
5,POC benchmark,1.206
6,Benchmark source,current-cycle portfolio median
7,Statistical conclusion,Credible observed underperformer
8,Observed conversations,111
9,Order-creation rate,63.1%


### Awareness Boost January

,Field,Value
0,Campaign type,awareness
1,Business objective,Create efficient qualified visibility before expecting sales.
2,Primary KPI,Reach
3,Primary KPI value,"636,884.000"
4,Primary KPI 95% interval,Not estimated
5,POC benchmark,"2,598,221.000"
6,Benchmark source,current-cycle portfolio median
7,Statistical conclusion,Point estimate only; uncertainty interval not estimated
8,Observed conversations,4
9,Order-creation rate,50.0%


### January Trial Bundle Test

...

,Field,Value
0,Campaign type,seasonal
1,Business objective,Capture time-sensitive demand during a limited campaign window.
2,Primary KPI,Net revenue per active day
3,Primary KPI value,"4,332.667"
4,Primary KPI 95% interval,"3,010.26 to 5,789.13"
5,POC benchmark,"1,916.667"
6,Benchmark source,current-cycle same-type median
7,Statistical conclusion,Credible observed outperformer
8,Observed conversations,47
9,Order-creation rate,87.2%


### Ramadan Iftar Premium Bundles

,Field,Value
0,Campaign type,seasonal
1,Business objective,Capture time-sensitive demand during a limited campaign window.
2,Primary KPI,Net revenue per active day
3,Primary KPI value,"2,566.600"
4,Primary KPI 95% interval,"1,936.93 to 3,238.73"
5,POC benchmark,"2,799.700"
6,Benchmark source,current-cycle same-type median
7,Statistical conclusion,Inconclusive relative to the POC benchmark
8,Observed conversations,84
9,Order-creation rate,83.3%


### Ramadan Suhoor Specials

,Field,Value
0,Campaign type,seasonal
1,Business objective,Capture time-sensitive demand during a limited campaign window.
2,Primary KPI,Net revenue per active day
3,Primary KPI value,"1,266.733"
4,Primary KPI 95% interval,"961.66 to 1,623.13"
5,POC benchmark,"3,449.633"
6,Benchmark source,current-cycle same-type median
7,Statistical conclusion,Credible observed underperformer
8,Observed conversations,64
9,Order-creation rate,75.0%


## 13. Final interpretation

The notebook can identify campaigns with attractive observed performance, but the
correct POC conclusion is evidence-aware:

- A high point estimate with a wide interval is a **test candidate**, not a winner.
- A narrow interval above a meaningful target is a stronger scaling candidate.
- A narrow interval below an acceptable target supports stopping or redesigning the
  campaign.
- A peer median is not a substitute for an approved business target.
- Because Meta and WhatsApp event populations are unreconciled, all WhatsApp-based
  findings describe the supplied sample and the budget remains illustrative.

The next analytical layer can apply the same pattern within each approved campaign:
adset, audience, creative, and ad point estimates, uncertainty, within-campaign peer
comparison, and controlled-test recommendations.